# 实验一：静态页面与 API 数据采集

**学时：4 学时**  ·  **平台：SmartPark DataLab**  ·  **代码文件：`lab1_html_api.ipynb`**

两条采集链路：

```text
Device  ：动态加载 → 找到 API → JSON → 分页采集全部设备（5000 条）
Company ：静态加载 → HTML → BeautifulSoup → "下一页" → 采集 3 页（60 条）
```

**运行环境**：课程实验环境 Docker 容器内的 JupyterLab，已装好 `requests` / `beautifulsoup4`。


## 0. 环境确认

先确认依赖可用、实验平台可达。这一步失败，后面全都不用看。


In [4]:
import json
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

# 实验平台地址（讲义指定）
BASE_URL = "https://115.159.159.82"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; SmartPark-DataLab/1.0)"}

sess = requests.Session()
sess.headers.update(HEADERS)

r = sess.get(BASE_URL + "/", timeout=10)
print("平台状态码:", r.status_code)
print("实验平台可用:", r.status_code == 200)


平台状态码: 200
实验平台可用: True


## 2. 先判断：静态加载还是动态加载

判断方法（讲义 2.1）：

```text
右键 → 查看网页源代码 → Ctrl+F 搜索页面上已看到的一条目标数据
  能搜到   → 静态加载 → requests + BeautifulSoup
  搜不到   → 动态加载 → F12 → Network → Fetch/XHR 找 API
```

下面用代码复现这个判断过程：直接看**初始 HTML**里有没有目标数据。


In [5]:
devices_html = sess.get(BASE_URL + "/devices", timeout=10).text
companies_html = sess.get(BASE_URL + "/companies", timeout=10).text

print("=== /devices（设备资产）===")
print("初始 HTML 长度:", len(devices_html))
print("含设备编号 D00000001 :", "D00000001" in devices_html)
print("含设备表格 tbody 行  :", '<tr><td class="mono">' in devices_html)
print("→ 结论：数据不在初始 HTML 中，属于动态加载，需要找 API")

print()
print("=== /companies（园区企业）===")
print("初始 HTML 长度:", len(companies_html))
print("含企业编号 C000001  :", "C000001" in companies_html)
print("含 company-card 节点:", "company-card" in companies_html)
print("→ 结论：数据随 HTML 一起返回，属于静态加载，可用 BeautifulSoup 解析")


=== /devices（设备资产）===
初始 HTML 长度: 2949
含设备编号 D00000001 : False
含设备表格 tbody 行  : False
→ 结论：数据不在初始 HTML 中，属于动态加载，需要找 API

=== /companies（园区企业）===
初始 HTML 长度: 11331
含企业编号 C000001  : True
含 company-card 节点: True
→ 结论：数据随 HTML 一起返回，属于静态加载，可用 BeautifulSoup 解析


**结论**（与浏览器里"查看网页源代码"的观察一致）：

| 页面 | 加载方式 | 采集方式 |
|---|---|---|
| `/devices` | 动态（JS + API） | 找 API → `requests` 拿 JSON |
| `/companies` | 静态（HTML 直出） | `BeautifulSoup` 解析 |

设备页的动态加载逻辑在 `static/js/devices.js` 里，核心就一句：

```javascript
const res = await fetch(`/api/devices?${SP.query(form, page)}`);
```

`SP.query()`（`static/js/common.js`）负责拼参数：把筛选表单的非空值 + `page` + `page_size=50` 拼成查询串。


## 3. 任务一：采集 Device API

### 3.1 找到设备 API

浏览器里的操作：

```text
1. 打开 https://115.159.159.82/devices
2. 右键 → 查看网页源代码 → Ctrl+F 搜 "D00000001" → 搜不到（确认是动态加载）
3. F12 → Network → 选 Fetch/XHR 过滤器
4. 刷新页面（F5）
5. 列表里找到返回设备数据的请求，点开看 Headers / Payload / Response
```

在 Network 面板中记录到的信息：

| 项目 | 值 |
|---|---|
| Request URL | `https://115.159.159.82/api/devices?page=1&page_size=50` |
| Request Method | `GET` |
| Query 参数 | `page`（页码，从 1 开始）、`page_size`（每页条数，默认 50） |
| 可选筛选参数 | `company_id`、`device_type`、`status`、`zone` |
| Response | JSON |

**关键点**：能用浏览器直接打开 `https://115.159.159.82/api/devices?page=1&page_size=2` 看到 JSON，
说明这个接口不需要 Cookie、不需要 Token，`requests` 可以直接复现。

### 3.2 用 requests 复现第一页请求


In [6]:
api_url = BASE_URL + "/api/devices"       # 从 Network 面板抄下来的地址

r = sess.get(
    api_url,
    params={
        "page": 1,
        "page_size": 50,
    },
    timeout=10,
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))


status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [7]:
data = r.json()

print(type(data))
print(data.keys())


<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


> **注意**：`print(data)` 会打印全部 50 条记录，输出很长。先看**结构**（上面这格），
> 再看**字段**（下面这格），最后才看单条记录。这是看陌生接口的标准顺序。


In [8]:
# 只看结构，不看内容：把 items 换成一条
preview = dict(data)
preview["items"] = data["items"][:1]
print(json.dumps(preview, ensure_ascii=False, indent=2)[:1200])


{
  "page": 1,
  "page_size": 50,
  "total": 5000,
  "items": [
    {
      "device_id": "D00000001",
      "company_id": "C000001",
      "device_name": "循环泵-000001-001",
      "device_type": "pump",
      "model": "SP-PUM-800",
      "zone": "智造一区",
      "install_date": "2021-09-19",
      "rated_power_kw": 71.2,
      "sampling_interval_s": 30,
      "protocol": "mqtt",
      "status": "running",
      "updated_at": "2026-09-01T03:58:00"
    }
  ]
}


确认 JSON 中四个关键字段：

| JSON 字段 | 含义 | 本页值 |
|---|---|---|
| `page` | 当前页 | 1 |
| `page_size` | 每页条数 | 50 |
| `total` | 设备总数 | 5000 |
| `items` | 本页设备列表 | 50 条 |


In [9]:
print("当前页 :", data["page"])
print("每页条数:", data["page_size"])
print("设备总数:", data["total"])
print("本页条数:", len(data["items"]))

print()
print("一条真实设备记录：")
print(json.dumps(data["items"][0], ensure_ascii=False, indent=2))


当前页 : 1
每页条数: 50
设备总数: 5000
本页条数: 50

一条真实设备记录：
{
  "device_id": "D00000001",
  "company_id": "C000001",
  "device_name": "循环泵-000001-001",
  "device_type": "pump",
  "model": "SP-PUM-800",
  "zone": "智造一区",
  "install_date": "2021-09-19",
  "rated_power_kw": 71.2,
  "sampling_interval_s": 30,
  "protocol": "mqtt",
  "status": "running",
  "updated_at": "2026-09-01T03:58:00"
}


### 3.3 计算总页数并采集全部设备

**不要写死 100 页**，用 `total` 和 `page_size` 算出来：

```text
total_pages = ceil(total / page_size) = (total + page_size - 1) // page_size
```


In [10]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size   # 向上取整，不引入 math

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)


total: 5000
page_size: 50
total_pages: 100


分页采集：每一页拿到的 `items` 累加到 `devices` 里。

> `page_size` 有服务端上限：实测 **100 可用，200 返回 HTTP 422**。所以这里保持浏览器同款 `page_size=50`，
> 共 100 次请求。如果中途某页失败，要能发现——下面顺手加了状态码检查。


In [11]:
devices = []

for page in range(1, total_pages + 1):
    r = sess.get(
        api_url,
        params={
            "page": page,
            "page_size": page_size,
        },
        timeout=10,
    )
    r.raise_for_status()          # 非 200 直接抛出，别默默丢数据

    page_data = r.json()
    devices.extend(page_data["items"])

    if page % 20 == 0 or page == total_pages:
        print(f"已完成 {page}/{total_pages} 页，累计 {len(devices)} 条")

print()
print("device count:", len(devices))


已完成 20/100 页，累计 1000 条
已完成 40/100 页，累计 2000 条
已完成 60/100 页，累计 3000 条
已完成 80/100 页，累计 4000 条
已完成 100/100 页，累计 5000 条

device count: 5000


In [12]:
# 字段含义对照
SAMPLE_FIELDS = {
    "device_id":          "设备编号，如 D00000001",
    "company_id":         "所属企业编号，如 C000001",
    "device_name":        "设备名称",
    "device_type":        "设备类型：pump / compressor / fan / motor / heat_exchanger / storage_tank",
    "model":              "型号",
    "zone":               "所在区域",
    "install_date":       "安装日期",
    "rated_power_kw":     "额定功率（kW）",
    "sampling_interval_s":"采样间隔（秒）",
    "protocol":           "通信协议，如 mqtt",
    "status":             "运行状态：running / stopped / maintenance / fault",
    "updated_at":         "数据更新时间",
}
for k, v in SAMPLE_FIELDS.items():
    print(f"{k:<22}{v}")

print()
assert len(devices) == 5000, f"设备总数不是 5000，实际 {len(devices)}"
print("✅ 目标达成：Device 总数 5000")


device_id             设备编号，如 D00000001
company_id            所属企业编号，如 C000001
device_name           设备名称
device_type           设备类型：pump / compressor / fan / motor / heat_exchanger / storage_tank
model                 型号
zone                  所在区域
install_date          安装日期
rated_power_kw        额定功率（kW）
sampling_interval_s   采样间隔（秒）
protocol              通信协议，如 mqtt
status                运行状态：running / stopped / maintenance / fault
updated_at            数据更新时间

✅ 目标达成：Device 总数 5000


## 4. 任务二：采集 Company 静态页面

打开 `https://115.159.159.82/companies`，用「查看网页源代码」搜索企业名称，能搜到 → 静态页面。


In [13]:
current_url = BASE_URL + "/companies"

r = sess.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

print("status:", r.status_code)
print("页面标题:", soup.title.get_text(strip=True))


status: 200
页面标题: 园区企业 · SmartPark DataLab


### 4.1 设计企业记录的 CSS Selector

先在 Elements 中找到**一条完整企业记录的最外层节点**，它的 `outerHTML` 长这样：

```html
<article class="company-card" data-company-id="C000001">
  <div class="company-card-top">
    <span class="mono">C000001</span>
    <span class="badge active">运营中</span>
  </div>
  <h2><a href="/companies/C000001">临港精工001有限公司</a></h2>
  <dl>
    <div><dt>所属行业</dt><dd>汽车零部件</dd></div>
    <div><dt>所在区域</dt><dd>智造一区</dd></div>
    <div><dt>员工规模</dt><dd>88 人</dd></div>
    <div><dt>设备数量</dt><dd>6 台</dd></div>
  </dl>
  <a class="detail-link" href="/companies/C000001">查看企业档案 →</a>
</article>
```

按讲义的顺序检查：

```text
① 有稳定的 id 吗？           → 没有
② 有稳定、够明确的 class 吗？ → 有：class="company-card"（语义明确，页面上每条记录都带）
③ 有 data-* 属性吗？          → 有：data-company-id="C000001"（用来取 company_id，比解析文本更稳）
④ 需要父子层级定位吗？        → 不需要
⑤ 需要位置关系吗？            → 不需要
```

所以 selector 就用最短的那一版：**`article.company-card`**（标签 + class，避免误匹配别的 `company-card` 元素）。
浏览器 Copy selector 通常会给出 `body > main > section.company-grid > article:nth-child(1)` 这种长路径——
`nth-child(1)` 只对第一条成立，必须简化掉。


In [14]:
cards = soup.select("article.company-card")

print("本页企业数：", len(cards))
assert len(cards) == 20


本页企业数： 20


### 4.2 提取企业字段

**先定位一条记录，再在记录内部提取字段。** 行业/区域/员工规模/设备数量是 `dt/dd` 结构，
先转成字典再取值。


In [15]:
card = cards[0]

print("data-company-id 属性：", card.get("data-company-id"))
print("整条记录的文本：")
print(card.get_text(" ", strip=True))


data-company-id 属性： C000001
整条记录的文本：
C000001 运营中 临港精工001有限公司 所属行业 汽车零部件 所在区域 智造一区 员工规模 88 人 设备数量 6 台 查看企业档案 →


In [16]:
# dt/dd → 字典
details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)


{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [17]:
def parse_company(card):
    """把一条 <article class="company-card"> 解析成字典。"""
    # dt/dd 明细 → 字典
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    # 设备数量："6 台" → 6
    device_count = int(details["设备数量"].replace("台", "").strip())

    return {
        "company_id": card["data-company-id"],            # data-* 属性最稳，不用解析文本
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "employee_count": int(details["员工规模"].replace("人", "").strip()),
        "device_count": device_count,
        "status": card.select_one(".badge").get_text(strip=True),   # 运营中 / 已停用
        "detail_url": urljoin(BASE_URL, card.select_one("h2 a")["href"]),
    }


print(json.dumps(parse_company(cards[0]), ensure_ascii=False, indent=2))


{
  "company_id": "C000001",
  "name": "临港精工001有限公司",
  "industry": "汽车零部件",
  "zone": "智造一区",
  "employee_count": 88,
  "device_count": 6,
  "status": "运营中",
  "detail_url": "https://115.159.159.82/companies/C000001"
}


In [18]:
companies = [parse_company(c) for c in cards]

print("本页企业数：", len(companies))
assert len(companies) == 20


本页企业数： 20


## 5. 任务三：根据"下一页"连续采集 3 页 Company

页面底部的分页条是：

```html
<nav class="pagination" aria-label="企业分页">
  <span>上一页</span><b>1</b><a href="/companies?page=2">下一页</a>
</nav>
```

第 2 页起，"上一页"也会变成 `<a>`；到**最后一页（第 15 页）**，"下一页"会退化成 `<span>`：

```html
<nav class="pagination">…<b>15</b><span>下一页</span></nav>
```

所以：

| 写法 | 第 1 页 | 第 2 页 | 第 15 页 | 可用性 |
|---|---|---|---|---|
| `nav.pagination a` | 下一页 ✅ | 拿到**上一页** ❌ | NaN | ✗ |
| `nav.pagination a:last-child` | 下一页 ✅ | 下一页 ✅ | None ✅ | ✓ |

用 `a:last-child`——它永远指向最右边那个链接；元素不是 `<a>` 时返回 `None`，正好当循环终止条件。


In [20]:
next_link = soup.select_one("nav.pagination a:last-child")

print("文本:", next_link.get_text(" ", strip=True))
print("href:", next_link.get("href"))

next_url = urljoin(current_url, next_link.get("href"))   # 相对路径 → 绝对路径
print("绝对地址:", next_url)


文本: 下一页
href: /companies?page=2
绝对地址: https://115.159.159.82/companies?page=2


把「请求当前页 → 解析本页企业 → 找到下一页 href」组合成循环，连续采集 **3 页**。


In [21]:
companies = []
current_url = BASE_URL + "/companies"

MAX_PAGES = 3

for page_no in range(1, MAX_PAGES + 1):
    r = sess.get(current_url, timeout=10)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    # ① 解析本页企业
    page_companies = [parse_company(c) for c in soup.select("article.company-card")]
    companies.extend(page_companies)
    print(f"第 {page_no} 页: {current_url} -> {len(page_companies)} 条")

    # ② 找下一页
    next_link = soup.select_one("nav.pagination a:last-child")
    if next_link is None:
        print("没有下一页了，停止")
        break
    current_url = urljoin(current_url, next_link["href"])

print()
print("Company 总数：", len(companies))
assert len(companies) == 60, f"企业总数不是 60，实际 {len(companies)}"
print("✅ 目标达成：3 页共 60 条")


第 1 页: https://115.159.159.82/companies -> 20 条
第 2 页: https://115.159.159.82/companies?page=2 -> 20 条
第 3 页: https://115.159.159.82/companies?page=3 -> 20 条

Company 总数： 60
✅ 目标达成：3 页共 60 条


## 6. 任务四：验证采集结果

### 6.1 数量与 ID 唯一性


In [22]:
print("companies:", len(companies))
print("unique company_id:", len(set(x["company_id"] for x in companies)))

print()

print("devices:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))

assert len(companies) == 60 and len(set(x["company_id"] for x in companies)) == 60
assert len(devices) == 5000 and len(set(x["device_id"] for x in devices)) == 5000
print()
print("✅ 数量与唯一性检查通过")


companies: 60
unique company_id: 60

devices: 5000
unique device_id: 5000

✅ 数量与唯一性检查通过


### 6.2 Company–Device 一致性

用**全部 Device** 的 `company_id` 统计每家企业的实际设备数，
跟前 60 家企业页面显示的 `device_count` 对比。对不上说明两个数据源不同步。


In [23]:
device_counts = Counter(x["company_id"] for x in devices)

mismatch = []
for c in companies:
    actual = device_counts.get(c["company_id"], 0)
    if actual != c["device_count"]:
        mismatch.append((c["company_id"], c["device_count"], actual))

print(f"参与比对的企业数: {len(companies)}")
print(f"不一致企业数: {len(mismatch)}")

if mismatch:
    print("前 5 条不一致明细（company_id, 页面显示, 实际统计）：")
    for row in mismatch[:5]:
        print("  ", row)
else:
    print("✅ 一致性检查通过：页面 device_count 与实际设备数完全吻合")

print()
print("抽查看一下：")
for c in companies[:3]:
    print(f"  {c['company_id']} {c['name']} — 页面 {c['device_count']} 台 / 实际 "
          f"{device_counts.get(c['company_id'], 0)} 台")


参与比对的企业数: 60
不一致企业数: 0
✅ 一致性检查通过：页面 device_count 与实际设备数完全吻合

抽查看一下：
  C000001 临港精工001有限公司 — 页面 6 台 / 实际 6 台
  C000002 临港装备002有限公司 — 页面 26 台 / 实际 26 台
  C000003 临港动力003有限公司 — 页面 6 台 / 实际 6 台


## 7. 任务五：保存采集结果

在 Notebook 同级目录建立 `data/`，保存为 JSON（本实验不引入 Pandas / 数据库 / Parquet）。


In [24]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")
print("当前工作目录:", Path.cwd())


数据已保存到 data/ 目录
当前工作目录: /home/pwuser/lab01


In [25]:
for name in ["devices.json", "companies.json"]:
    p = data_dir / name
    loaded = json.loads(p.read_text(encoding="utf-8"))
    print(f"{p}  {p.stat().st_size / 1024:.1f} KB  {len(loaded)} 条")


data/devices.json  1862.9 KB  5000 条
data/companies.json  16.5 KB  60 条


In [26]:
print("Company 总数：", len(companies))
print("唯一 company_id：", len(set(x["company_id"] for x in companies)))
print("Device 总数：", len(devices))
print("唯一 device_id：", len(set(x["device_id"] for x in devices)))

from collections import Counter
device_counts = Counter(x["company_id"] for x in devices)
mismatch = sum(1 for c in companies
               if device_counts.get(c["company_id"], 0) != c["device_count"])
print("Company–Device 不一致企业数：", mismatch)


Company 总数： 60
唯一 company_id： 60
Device 总数： 5000
唯一 device_id： 5000
Company–Device 不一致企业数： 0


## 8. 完成前自检

```text
☑ 会用"查看网页源代码"判断静态 / 动态加载          → 第 1 节
☑ 能在 Network 中找到 Device API                  → 3.1  /api/devices
☑ 能读懂 API 分页结构并自动计算总页数              → 3.3  (total, page_size) -> 100 页
☑ Device 共采集 5000 条，device_id 无重复          → 任务四
☑ 能设计简洁、稳定的 Company CSS Selector          → 4.1  article.company-card
☑ 能定位"下一页"并读取 href                        → 任务三  nav.pagination a:last-child
☑ Company 连续采集 3 页，共 60 条                  → 任务三
☑ company_id 无重复                                → 任务四
☑ Company–Device 数量检查通过                      → 任务四
☑ 已保存 companies.json 和 devices.json            → 任务五
```

### 踩坑记录

1. **selector 别写太长**：Copy selector 给的是 `article:nth-child(1)`，只对第一条有效。
2. **"下一页"不能用 `nav.pagination a`**：第 2 页会先匹配到"上一页"。
3. **`page_size` 上限 100**：写 200 会返回 `HTTP 422`。
4. **"最后一页"的下一页是 `<span>`**：用 `a:last-child` 返回 `None` 自动终止。
5. **别写死页数/条数**：`total`、`page_size` 都是接口给的，算出来更稳。
6. **`raise_for_status()` 不能省**：否则某页 500 会被当成"这页没数据"，最后才发现少了几百条。
